In [2]:
# load virtual python environment with source ./.venv/bin/activate
# Install pyrpl

# ssh into red pitaya using...
# ssh root@rp-f0ea6b.local, or ssh 192.168.68.180
# default credentials are just 
# Cant rely on doing this in software as it won't be quick enough, which is the whole point of this board...
#Reload only modules imported with %aimport.

import os
#os.environ["QT_QPA_PLATFORM"] = "offscreen" # Needed for Ubuntu, weird wayland bug with pyrpl and PyQt5.

from pyrpl import Pyrpl

from matplotlib import pyplot as plt
import pitayaconfig
import time

import importlib

: 

In [1]:
myconfig = "1" 
hostname = pitayaconfig.IPADDRESS2 #IPADDRESS
gui = True

# create an interface to the Red Pitaya
p = Pyrpl(config=myconfig, hostname = hostname, gui=gui)

#r = Pyrpl(config=myconfig,hostname = pitayaconfig.HOSTNAME,gui=True).redpitaya

r = p.redpitaya
p.show_gui()


NameError: name 'pitayaconfig' is not defined

In [ ]:
# Testing piping the output of one IQ filter into the other to run frequency doubler


In [ ]:
# Sample the ADC voltage in a loop and plot the results

times, data = [],[]
t0 = time.time()
n = 3000
for i in range(n):
    times.append(time.time()-t0)
    data.append(r.scope.voltage_in1)
print("Rough time to read one FPGA register: ", (time.time()-t0)/n*1e6, "microseconds")
# matplotlib inline
f, axarr = plt.subplots(1,2, sharey=True)
axarr[0].plot(times, data, "+")
axarr[0].set_title("ADC voltage vs time")
axarr[1].hist(data, bins=10,density=True, orientation="horizontal")
axarr[1].set_title("ADC voltage histogram")

In [ ]:
# Signal generator example
asg = r.asg1 # make a shortcut to
print("Trigger sources:", asg.trigger_sources)
print("Output options: ", asg.output_directs)
asg.output_direct = 'out2'
asg.setup(waveform='sin',
        frequency=200,
        amplitude=0.001,
        offset=0,
        trigger_source='immediately')

In [ ]:
# Output DC from the first output... bias = offset
asg = r.asg1 # make a shortcut to
print("Trigger sources:", asg.trigger_sources)
print("Output options: ", asg.output_directs)
asg.output_direct = 'out1'
asg.setup(waveform="dc", offset = 1.0, trigger_source='immediately')

In [ ]:
#Oscilloscope example
s = r.scope # shortcut
asg = r.asg1

print("Available decimation factors:", s.decimations)
print("Trigger sources:", s.trigger_sources)
print("Available inputs: ", s.inputs)

# turn off asg so the scope has a chance to measure its "off-state" as well
asg.output_direct = "off"

# setup scope
s.input1 = 'asg1'

# pass asg signal through pid0 with a simple integrator - just for fun
# (detailed explanations for pid will follow)
r.pid0.input = 'asg1'
r.pid0.ival = 0 # reset the integrator to zero
r.pid0.i = 1000 # unity gain frequency of 1000 hz
r.pid0.p = 1.0 # proportional gain of 1.0
r.pid0.inputfilter = [0,0,0,0] # leave input filter disabled for now

# show pid output on channel2
s.input2 = 'pid0'

# trig at zero volt crossing
s.threshold = 0

# positive/negative slope is detected by waiting for input to
# sweep through hysteresis around the trigger threshold in
# the right direction
s.hysteresis = 0.01

# trigger on the input signal positive slope
s.trigger_source = 'ch1_positive_edge'

# take data symetrically around the trigger event
s.trigger_delay = 0

# set decimation factor to 64 -> full scope trace is 8ns * 2^14 * decimation = 8.3 ms long
s.decimation = 64

# launch a single (asynchronous) curve acquisition, the asynchronous
# acquisition means that the function returns immediately, eventhough the
# data-acquisition is still going on.
res = s.single_async()

print("Before turning on asg:")
print("Curve ready:", s.curve_ready()) # trigger should still be armed

# turn on asg and leave enough time for the scope to record the data
asg.setup(frequency=1e3,
        amplitude=0.3,
        start_phase=90,
        waveform='halframp',
        trigger_source='immediately')
time.sleep(0.010)

# check that the trigger has been disarmed
print("After turning on asg:")
print("Curve ready:", s.curve_ready())
print("Trigger event age [ms]:",8e-9*((
s.current_timestamp&0xFFFFFFFFFFFFFFFF) - s.trigger_timestamp)*1000)

# The function single_async returns a *future* (or promise) of the curve. To
# access the actual curve, use result()
ch1, ch2 = res.result()

# plot the data
# matplotlib inline
plt.plot(s.times*1e3, ch1, s.times*1e3, ch2)
plt.xlabel("Time [ms]")
plt.ylabel("Voltage")